# Task 1 – MNIST classification demo

This notebook demonstrates how to use the unified `MnistClassifier` wrapper with three algorithms:
- Random Forest (`"rf"`)
- Feed-Forward Neural Network (`"nn"`)
- Convolutional Neural Network (`"cnn"`)

It also includes a few edge cases:
- Input shape `(N, 28, 28)` vs `(N, 1, 28, 28)`
- Predicting on a single image
- Invalid input shapes raising a clear error


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

from torchvision import datasets, transforms

# Local project imports
from mnist_classifier import MnistClassifier


In [ ]:
# Download/load MNIST via torchvision
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="data", train=False, download=True, transform=transform)

# Use raw uint8 images for convenience; models normalize internally where needed
x_train = train_dataset.data.numpy().astype(np.float32)  # (N, 28, 28)
y_train = train_dataset.targets.numpy().astype(np.int64)

x_test = test_dataset.data.numpy().astype(np.float32)
y_test = test_dataset.targets.numpy().astype(np.int64)

x_train.shape, y_train.shape, x_test.shape, y_test.shape


In [ ]:
# To keep the demo fast, train on a subset.
# Increase these numbers for better accuracy.
n_train = 10000
n_test = 2000

x_train_s = x_train[:n_train]
y_train_s = y_train[:n_train]
x_test_s = x_test[:n_test]
y_test_s = y_test[:n_test]

x_train_s.shape, x_test_s.shape


## Random Forest baseline

In [ ]:
rf = MnistClassifier(algorithm="rf", n_estimators=200, random_state=42)
rf.train(x_train_s, y_train_s)
rf_preds = rf.predict(x_test_s)

acc_rf = accuracy_score(y_test_s, rf_preds)
acc_rf


## Feed-Forward Neural Network

In [ ]:
nn = MnistClassifier(algorithm="nn", epochs=3, batch_size=128, learning_rate=1e-3, hidden_dim=256)
nn.train(x_train_s, y_train_s)
nn_preds = nn.predict(x_test_s, batch_size=256)

acc_nn = accuracy_score(y_test_s, nn_preds)
acc_nn


## Convolutional Neural Network

In [ ]:
cnn = MnistClassifier(algorithm="cnn", epochs=3, batch_size=128, learning_rate=1e-3)
cnn.train(x_train_s, y_train_s)
cnn_preds = cnn.predict(x_test_s, batch_size=256)

acc_cnn = accuracy_score(y_test_s, cnn_preds)
acc_cnn


## Edge cases

In [ ]:
# 1) Input shape handling: (N, 28, 28) vs (N, 1, 28, 28)
batch = x_test_s[:8]
batch_ch = batch[:, None, :, :]

p1 = cnn.predict(batch)
p2 = cnn.predict(batch_ch)

print("Same predictions for different input shapes:", np.array_equal(p1, p2))
print(p1)


In [ ]:
# 2) Predict on a single image
single = x_test_s[:1]
single_pred = rf.predict(single)
single_pred, single_pred.shape


In [ ]:
# 3) Invalid shapes -> clear ValueError
try:
    bad = np.zeros((28, 28), dtype=np.float32)  # missing batch dimension
    _ = cnn.predict(bad)
except Exception as e:
    print(type(e).__name__, str(e))
